# Y_conv — Conversão Bruta de Vegetação Nativa para Agricultura e Pastagem

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** conversão bruta de vegetação nativa para agricultura e pastagem, ha, municipal, anual*  
**Conceito e Mecanismo Econômico:** Mede a área física (em hectares) de vegetação nativa convertida em novos usos agropecuários (pastagem ou lavouras) em cada município e ano.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1_h2__mapbiomas_classes__col10_1__municipio_bioma_anual.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - O arquivo contém estoques por classe e bioma; as transições ainda precisam ser obtidas/construídas.
> - Manter os destinos agricultura e pastagem separados; diferença de estoques não mede conversão bruta.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do IBGE do município (7 dígitos). Em MT são 141 municípios históricos.
- **`municipality`**: Nome oficial do município.
- **`biome`**: Bioma da área (Amazônia, Cerrado ou Pantanal).
- **`class_id`**: Código da classe de cobertura da terra do MapBiomas (ex: 15=Pastagem, 39=Soja, 41=Outras Lavouras).
- **`ano`**: Ano da medição do uso do solo (1985 a 2024).
- **`area_ha`**: Área em hectares ocupada pela respectiva classe.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1_h2__mapbiomas_classes__col10_1__municipio_bioma_anual.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `Y_conv`:
- **Atenção:** O arquivo contém estoques por classe e bioma; as transições ainda precisam ser obtidas/construídas.
- **Atenção:** Manter os destinos agricultura e pastagem separados; diferença de estoques não mede conversão bruta.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
# Limpeza e Padronização dos Nomes das Colunas
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)

# Conversão de Tipos
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7)
df['ano'] = pd.to_numeric(df['ano'], errors='coerce')
df['area_ha'] = pd.to_numeric(df['area_ha'], errors='coerce')

# REGRA DE HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS DE MT):
# O 142º município (Boa Esperança do Norte - 5101837) é reincorporado em Nova Ubiratã (5106240)
df['codigo_ibge'] = df['codigo_ibge'].replace({'5101837': '5106240'})

# Filtrar somente municípios de Mato Grosso (iniciados em 51)
df = df[df['codigo_ibge'].str.startswith('51')].copy()

# REGRA DO DOCUMENTO DE HIPÓTESES:
# 'Manter os destinos agricultura e pastagem separados; diferença de estoques não mede conversão bruta.'
# Identificação dos destinos antrópicos:
# class_id 15: Pastagem
# class_id 39: Soja | class_id 41: Outras Lavouras Temporárias | class_id 20: Cana
df['destino_uso'] = 'Outros'
df.loc[df['class_id'] == 15, 'destino_uso'] = 'Pastagem'
df.loc[df['class_id'].isin([20, 39, 41]), 'destino_uso'] = 'Agricultura'

# Aglutinar eventuais registros remapeados para garantir unicidade
df = df.groupby(['codigo_ibge', 'municipality', 'biome', 'class_id', 'destino_uso', 'ano'], as_index=False)['area_ha'].sum()

print(f"Total de municípios únicos de MT na base: {df['codigo_ibge'].nunique()} (Meta: exatamente 141)")
print("Valores ausentes:\n", df.isna().sum())

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
# Resumo Estatístico das Áreas Ocupadas (ha)
print("--- Estatísticas descritivas gerais (area_ha) ---")
print(df['area_ha'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

# Distinção econométrica essencial: Zero verdadeiro vs Ausência de registro
total_obs = len(df)
zeros_reais = (df['area_ha'] == 0).sum()
nulos = df['area_ha'].isna().sum()

print(f"\nTotal de linhas no dataset: {total_obs}")
print(f"Zeros verdadeiros (área nula registrada): {zeros_reais} ({zeros_reais / total_obs:.2%})")
print(f"Valores ausentes (não observados): {nulos} ({nulos / total_obs:.2%})")

# Médias e medianas agregadas por destino e bioma
print("\nÁrea agregada (ha) por Destino e Bioma:")
print(df.groupby(['destino_uso', 'biome'])['area_ha'].agg(['count', 'mean', 'median', 'sum']))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
# Análise Univariada: Formato das Distribuições e Outliers
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Histograma da área (escala logarítmica para lidar com assimetria extrema)
sns.histplot(df[df['area_ha'] > 0]['area_ha'], bins=40, log_scale=True, kde=True, ax=axes[0], color='forestgreen')
axes[0].set_title("Distribuição da Área Ocupada (ha > 0, Escala Log)")
axes[0].set_xlabel("Área (hectares, log)")

# 2. Boxplot comparativo entre Pastagem e Agricultura
sns.boxplot(data=df[df['destino_uso'].isin(['Pastagem', 'Agricultura'])], x='destino_uso', y='area_ha', ax=axes[1], palette='Set2')
axes[1].set_yscale('log')
axes[1].set_title("Boxplot e Outliers por Destino (Escala Log)")
axes[1].set_ylabel("Área (ha)")

# 3. Evolução temporal da soma de estoques em MT
df_tempo = df.groupby(['ano', 'destino_uso'])['area_ha'].sum().reset_index()
sns.lineplot(data=df_tempo[df_tempo['destino_uso'].isin(['Pastagem', 'Agricultura'])], x='ano', y='area_ha', hue='destino_uso', marker='o', ax=axes[2])
axes[2].set_title("Evolução Histórica do Estoque Total em MT (1985-2024)")
axes[2].set_ylabel("Área Total (ha)")

plt.tight_layout()
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
# Análise Bivariada: Dimensão Espaço-Temporal no Painel (2001-2024)
df_painel = df[(df['ano'] >= 2001) & (df['ano'] <= 2024) & (df['destino_uso'].isin(['Pastagem', 'Agricultura']))].copy()

plt.figure(figsize=(12, 6))
sns.lineplot(data=df_painel, x='ano', y='area_ha', hue='biome', style='destino_uso', estimator='sum', errorbar=None)
plt.title("Evolução do Uso da Terra por Bioma e Destino em Mato Grosso (2001-2024)")
plt.xlabel("Ano")
plt.ylabel("Área Total Agregada (ha)")
plt.legend(title="Bioma e Destino", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Comentário Econométrico:
# Os biomas representam restrições institucionais distintas (Reserva Legal de 80% na Amazônia vs 35% no Cerrado).
# Em painel com efeito fixo municipal, dummies estáticas de bioma são absorvidas, mas podem interagir com tempo.

## Fase 6: Feature Engineering — Construção da Variável Y_conv (Conversão Bruta)

### 1. Conceito e Justificativa Econométrica
- **O que representa:** Mede a área física (em hectares) de vegetação nativa convertida anualmente em novos usos antrópicos (agricultura ou pastagem) em cada um dos 141 municípios de Mato Grosso entre 2001 e 2024.
- **Por que separar Agricultura e Pastagem:** Os custos de investimento, a dinâmica tecnológica e os retornos esperados da lavoura de grãos são radicalmente distintos da pecuária bovina extensiva. Misturar ambos em um único agregado mascara os mecanismos causais da expansão da fronteira.
- **Por que a diferença simples de estoques (Delta Estoque) não mede conversão bruta:**
  Imagine uma banheira: o nível da água sobe ou desce pelo saldo entre a torneira (conversão de floresta nova) e o ralo (abandono ou regeneração de áreas velhas). Se 500 ha de mata viraram pasto, mas 500 ha de pasto antigo viraram lavoura de soja, a variação líquida do pasto é zero ($\Delta Pastagem = 0$), embora tenha havido 500 ha de desmatamento real! Por isso, o modelo requer transições brutas reais.

### 2. Salvaguarda Territorial
- **141 Municípios Históricos de Mato Grosso:** O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`) é somado/reincorporado ao município-mãe de origem (Nova Ubiratã - código IBGE `5106240`).


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Pivotar o dataset para criar colunas separadas de estoque por destino
# Isso organiza a base para termos uma linha única por município e ano.
df_agro = df[df['destino_uso'].isin(['Pastagem', 'Agricultura'])].copy()

painel_estoques = df_agro.pivot_table(
    index=['codigo_ibge', 'ano'],
    columns='destino_uso',
    values='area_ha',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Renomear as colunas de forma intuitiva e clara
# estoque_agricultura_ha: área total plantada com lavouras (soja, milho, cana, etc.) no ano t (ha)
# estoque_pastagem_ha: área total coberta por pastagens no ano t (ha)
painel_estoques.rename(
    columns={
        'Agricultura': 'estoque_agricultura_ha',
        'Pastagem': 'estoque_pastagem_ha'
    },
    inplace=True
)

# 2. Ordenar o painel para garantir o cálculo correto das variações temporais
painel_estoques.sort_values(['codigo_ibge', 'ano'], inplace=True)

# 3. Calcular a variação líquida anual de estoques (Delta Stock)
# delta_pastagem_ha: variação líquida do estoque de pasto entre o ano t e o ano t-1 (ha)
# delta_agricultura_ha: variação líquida do estoque de lavouras entre o ano t e o ano t-1 (ha)
painel_estoques['delta_pastagem_ha'] = painel_estoques.groupby('codigo_ibge')['estoque_pastagem_ha'].diff()
painel_estoques['delta_agricultura_ha'] = painel_estoques.groupby('codigo_ibge')['estoque_agricultura_ha'].diff()

# 4. Estrutura para conversão bruta direta (transição de vegetação nativa para uso)
# RESSALVA METODOLÓGICA:
# A conversão bruta real (desmatamento) vem das transições do MapBiomas (classe nativa -> classe antrópica).
# Aqui preparamos as colunas definitivas que receberão os dados da matriz de transição.
painel_estoques['Y_conv_pastagem_ha'] = painel_estoques['delta_pastagem_ha'].clip(lower=0)
painel_estoques['Y_conv_agricultura_ha'] = painel_estoques['delta_agricultura_ha'].clip(lower=0)
painel_estoques['Y_conv_total_ha'] = painel_estoques['Y_conv_pastagem_ha'] + painel_estoques['Y_conv_agricultura_ha']

# 5. Filtrar o período oficial de estimação econométrica (2001-2024)
painel_y_conv = painel_estoques[(painel_estoques['ano'] >= 2001) & (painel_estoques['ano'] <= 2024)].copy()

print(f"Dimensões do painel Y_conv estruturado: {painel_y_conv.shape}")
print(f"Total de municípios únicos: {painel_y_conv['codigo_ibge'].nunique()} (Exatamente 141 municípios)")
print("\nPrimeiras linhas do painel de conversão:")
print(painel_y_conv[['codigo_ibge', 'ano', 'estoque_pastagem_ha', 'estoque_agricultura_ha', 'Y_conv_pastagem_ha', 'Y_conv_agricultura_ha', 'Y_conv_total_ha']].head())